In [19]:
import urllib
from tqdm import tqdm
import requests
import pandas as pd
import re
import json

## This notebook adds which politician it is, and what their party is


In [3]:
with urllib.request.urlopen(f"https://raw.githubusercontent.com/Somon8/social_graphs_25/main/final-project/voting-data/df_votes_all_periods.csv") as response:
    df = pd.read_csv(response)

df.head(2)

,metadata_vote_id,vote_type,voting_id,aktør_id,metadata_opdateringsdato
0,1,1,1,158,2014-09-09T09:05:59.653
1,2,1,1,71,2014-09-09T09:05:59.653


In [23]:
with urllib.request.urlopen(f"https://raw.githubusercontent.com/Somon8/social_graphs_25/main/final-project/party_colors.json") as response:
    party_colors = json.load(response)

In [ ]:
# party_colors
# parties = []
# for party in party_colors:
#     parties.append()

ValueError: too many values to unpack (expected 2)

In [ ]:
request_session = requests.Session()
def get_actors_for_votes(aktør_id,  session = request_session):
    base_url = "https://oda.ft.dk/api/"
    item = "Aktør"
    url = f"{base_url}{item}({aktør_id})"
    response = session.get(url)
    # print(f"Sent call to URL: {response.url}")
    if response.status_code != 200: #If the response is not ok print it and continue, no need to break the program.
        print(f"HTTP error for {aktør_id}: ", response.status_code)
        print("Response text:", response.text)
        return None
    else:
        try:
            data = response.json()
        except ValueError:
            print("Error: Response is not valid JSON")
            print("Response text:", response.text)
            return None
        
    politician_name = data.get("navn")
    try:
        politician_standard_party = re.search('<party>([^<]+)</party>',  data.get("biografi")).group(1)
    except:
        politician_standard_party = "Not able to assign"
        print(f"Not able to assign party for {politician_name} with id {aktør_id}")
        
    return politician_name, politician_standard_party

In [ ]:
unique_actors = df['aktør_id'].unique()

all_actors = []
for aktør_id in tqdm(unique_actors):
    data = get_actors_for_votes(aktør_id)
    all_actors.append(data)
actor_df = pd.DataFrame(all_actors)
actor_df.rename(columns = {0 : "politician", 1: "party"}, inplace = True)

  0%|          | 0/710 [00:00<?, ?it/s]

 46%|████▋     | 329/710 [00:10<00:09, 39.85it/s]

Not able to assign party for Carsten Hansen with id 5593
Not able to assign party for Torben Hansen with id 7633
Not able to assign party for Mogens Jensen, Brøndby with id 5905


 61%|██████    | 430/710 [00:12<00:07, 39.71it/s]

Not able to assign party for Jytte Andersen with id 8319


 66%|██████▌   | 467/710 [00:13<00:06, 40.07it/s]

Not able to assign party for Frode Sørensen, Hjørring with id 3042


 73%|███████▎  | 521/710 [00:15<00:04, 38.57it/s]

Not able to assign party for Aage Frandsen with id 1623


100%|██████████| 710/710 [00:20<00:00, 35.00it/s]


In [ ]:
actor_df

,politician,party
0,Eigil Andersen,Socialistisk Folkeparti
1,Tom Behnke,Det Konservative Folkeparti
2,Liselott Blixt,Uden for folketingsgrupperne
3,Erling Bonnesen,Venstre
4,Bent Bøgsted,Danmarksdemokraterne
...,...,...
705,Lisa Perkins,Liberal Alliance
706,Joachim Hoffmann-Petersen,Det Konservative Folkeparti
707,Erik Veje Rasmussen,Venstre
708,Mads Madsen Henriksen,Socialdemokratiet


In [16]:
request_session = requests.Session()
def get_actor_party(aktør_id,  session = request_session):
    base_url = "https://oda.ft.dk/api/"
    item = "AktørAktør"
    url = f"{base_url}{item}({aktør_id})"
    response = session.get(url)
    # print(f"Sent call to URL: {response.url}")
    if response.status_code != 200: #If the response is not ok print it and continue, no need to break the program.
        print(f"HTTP error for {aktør_id}: ", response.status_code)
        print("Response text:", response.text)
        return None
    else:
        try:
            data = response.json()
        except ValueError:
            print("Error: Response is not valid JSON")
            print("Response text:", response.text)
            return None
        
    return data

get_actor_party(unique_actors[100])

HTTP error for 62:  404
Response text: 


In [ ]:
#AktørAktørRolle = 15 -> personen er medlem af partiet
#fraaktørid = 158
# Giver medlem (rolle15) af: 1259=SF, 1088=SF, 1315=SF, 1165=SF, 6=Folketinget.
# Partierne har typeid=4
# Vi skal altså filtere på typeid = 4

# tilaktørid = aktør_id
# For hvert id skal vi finde alle AktørAktør relationer med rolleid = 15
# Så skal vi tage startdato og slutdato
# Så skal vi på fraaktørid finde navnet på den aktør, som så vil være et parti
# Så skal vi lave en eller anden smart gruppering så det bliver joinet på dataen

